In [2]:
import pyarrow.parquet as pq

path = '/Users/mariammohamed/Documents/uni/sem8/grad/GlucoTwin_New/datasets/metabonet_split/Loop.parquet'
pf = pq.ParquetFile(path)

total_rows = pf.metadata.num_rows
null_counts = {col: 0 for col in pf.schema_arrow.names}

# Read in manageable batches
for batch in pf.iter_batches(batch_size=100_000):
    for col in batch.schema.names:
        null_counts[col] += batch.column(col).null_count

print("=" * 80)
print(f"Loop — EMPTY VALUES PER COLUMN")
print(f"Total rows: {total_rows:,}")
print("=" * 80)

results = []

for col, null_count in null_counts.items():
    percentage = (null_count / total_rows) * 100

    results.append((col, null_count, percentage))

# Sort from most empty to least empty
results.sort(key=lambda x: x[2], reverse=True)

for col, null_count, percentage in results:
    print(f"{col:<40} {null_count:>12,} empty | {percentage:>8.2f}%")

Loop — EMPTY VALUES PER COLUMN
Total rows: 94,810,573
is_pregnant                                45,902,354 empty |    48.41%
carbs                                      15,347,962 empty |    16.19%
CGM                                        10,175,566 empty |    10.73%
bolus                                       9,925,768 empty |    10.47%
insulin_type_basal                          9,178,209 empty |     9.68%
insulin_type_bolus                          9,178,209 empty |     9.68%
basal                                       9,039,245 empty |     9.53%
insulin                                     8,936,497 empty |     9.43%
ethnicity                                   7,909,608 empty |     8.34%
weight                                      7,732,368 empty |     8.16%
age_of_diagnosis                            7,603,314 empty |     8.02%
height                                      7,487,761 empty |     7.90%
insulin_delivery_device                     7,352,208 empty |     7.75%
cgm_device

# time checkings

nan = not recorded 

In [4]:
#This code displays the first 100 rows of the first row group in Loop.parquet, showing only the seven columns you selected.
import pyarrow.parquet as pq

path = "/Users/mariammohamed/Documents/uni/sem8/grad/GlucoTwin_New/datasets/metabonet_split/Loop.parquet"

pf = pq.ParquetFile(path)

columns = [
    "date",
    "CGM",
    "carbs",
    "basal",
    "insulin",
    "bolus",
    "id"
]

table = pf.read_row_group(0, columns=columns)

df = table.to_pandas().head(100)

print(df.to_string(index=False))

               date   CGM  carbs    basal  insulin  bolus id
2018-12-20 19:05:00 161.0    NaN      NaN      NaN    NaN 10
2018-12-20 19:10:00 160.0    NaN      NaN      NaN    NaN 10
2018-12-20 19:15:00 158.0    NaN      NaN      NaN    NaN 10
2018-12-20 19:20:00 154.0    NaN      NaN      NaN    NaN 10
2018-12-20 19:25:00 152.0    NaN      NaN      NaN    NaN 10
2018-12-20 19:30:00 147.0    NaN      NaN      NaN    NaN 10
2018-12-20 19:35:00 142.0    NaN      NaN      NaN    NaN 10
2018-12-20 19:40:00 137.0    NaN      NaN      NaN    NaN 10
2018-12-20 19:45:00 133.0    NaN      NaN      NaN    NaN 10
2018-12-20 19:50:00 134.0    NaN      NaN      NaN    NaN 10
2018-12-20 19:55:00 137.0    NaN      NaN      NaN    NaN 10
2018-12-20 20:00:00 143.0    NaN      NaN      NaN    NaN 10
2018-12-20 20:05:00 153.0    NaN      NaN      NaN    NaN 10
2018-12-20 20:10:00 170.0    NaN      NaN      NaN    NaN 10
2018-12-20 20:15:00 186.0    NaN      NaN      NaN    NaN 10
2018-12-20 20:20:00 198.

# understanding carbs=0

NaN → no carbohydrate event recorded
////// 0   → carbohydrate event with 0 g

In [6]:
import pyarrow.parquet as pq

path = "/Users/mariammohamed/Documents/uni/sem8/grad/GlucoTwin_New/datasets/metabonet_split/Loop.parquet"

pf = pq.ParquetFile(path)

columns = [
    "date",
    "CGM",
    "carbs",
    "basal",
    "insulin",
    "bolus",
    "id"
]

# Read first row group only
table = pf.read_row_group(0, columns=columns)
df = table.to_pandas()

print("Rows in row group:", len(df))
print("\nNon-null counts:")
print(df.notna().sum())

print("\nUnique carbs values:")
print(df["carbs"].value_counts(dropna=False).head(20))

print("\nUnique bolus values:")
print(df["bolus"].describe())

print("\nRows containing bolus:")
print(
    df[df["bolus"].notna()][
        ["date", "CGM", "carbs", "basal", "insulin", "bolus", "id"]
    ].head(30).to_string(index=False)
)

Rows in row group: 96456

Non-null counts:
date       96456
CGM        86608
carbs      70841
basal      96420
insulin    96420
bolus      96294
id         96456
dtype: int64

Unique carbs values:
carbs
0.0     69447
NaN     25615
15.0      274
20.0      160
25.0      135
10.0      123
18.0      103
12.0       63
30.0       52
8.0        51
13.0       48
23.0       44
35.0       39
28.0       34
16.0       29
17.0       27
22.0       26
40.0       20
24.0       19
5.0        16
Name: count, dtype: int64

Unique bolus values:
count    96294.000000
mean         0.033815
std          0.250401
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          5.000000
Name: bolus, dtype: float64

Rows containing bolus:
               date  CGM  carbs    basal  insulin  bolus id
2018-12-21 08:35:00 63.0    NaN 0.066667 1.716667   1.65 10
2018-12-21 08:40:00 60.0    NaN 0.066667 0.066667   0.00 10
2018-12-21 08:45:00 59.0    NaN 0.066667 0.066667   0.00 10
2

In [7]:
import pyarrow.parquet as pq

path = "/Users/mariammohamed/Documents/uni/sem8/grad/GlucoTwin_New/datasets/metabonet_split/Loop.parquet"

pf = pq.ParquetFile(path)

# Read only the first 10 rows
table = pf.iter_batches(batch_size=300).__next__()
df = table.to_pandas()

print(df.to_string(index=False))

  CGM  age    basal  bolus  carbs cgm_device                date   height id  insulin insulin_delivery_algorithm insulin_delivery_device insulin_delivery_modality insulin_type_basal insulin_type_bolus  is_pregnant  is_test source_file  weight gender  age_of_diagnosis ethnicity  subject_split_across_traintest patient_stream_id
161.0 32.0      NaN    NaN    NaN  Dexcom G6 2018-12-20 19:05:00 5.333333 10      NaN              LoopAlgorithm                 OmniPod                       AID   Humalog (Lispro)   Humalog (Lispro)        False    False        Loop   143.0 Female              14.0     White                           False           Loop_10
160.0 32.0      NaN    NaN    NaN  Dexcom G6 2018-12-20 19:10:00 5.333333 10      NaN              LoopAlgorithm                 OmniPod                       AID   Humalog (Lispro)   Humalog (Lispro)        False    False        Loop   143.0 Female              14.0     White                           False           Loop_10
158.0 32.0      

In [ ]:
#dropping columns from Loop.parquet and saving to Loopdropped.parquet
import pyarrow.parquet as pq
import pyarrow as pa

input_path = '/Users/mariammohamed/Documents/uni/sem8/grad/GlucoTwin_New/datasets/metabonet_split/Loop.parquet'

output_path = '/Users/mariammohamed/Documents/uni/sem8/grad/GlucoTwin_New/datasets/metabonet_split/Loopdropped.parquet'

# Columns to drop
drop_cols = [
    "is_pregnant",
    "ethnicity",
    "insulin_delivery_device",
    "cgm_device",
    "is_test",
    "source_file",
    "subject_split_across_traintest",
    "patient_stream_id",
]

pf = pq.ParquetFile(input_path)

# Get remaining columns
remaining_cols = [
    col for col in pf.schema.names
    if col not in drop_cols
]

print("Columns being kept:")
for col in remaining_cols:
    print(" -", col)

print("\nColumns being dropped:")
for col in drop_cols:
    print(" -", col)

# Create writer using the original schema minus dropped columns
first_table = pf.read_row_group(0, columns=remaining_cols)

writer = pq.ParquetWriter(
    output_path,
    first_table.schema,
    compression="snappy"
)

# Write first row group
writer.write_table(first_table)

# Write remaining row groups
for i in range(1, pf.num_row_groups):
    table = pf.read_row_group(i, columns=remaining_cols)
    writer.write_table(table)

    print(f"Processed row group {i + 1}/{pf.num_row_groups}")

writer.close()

print("\nDone!")
print(f"Saved to:\n{output_path}")

Columns being kept:
 - CGM
 - age
 - basal
 - bolus
 - carbs
 - date
 - height
 - id
 - insulin
 - insulin_delivery_algorithm
 - insulin_delivery_modality
 - insulin_type_basal
 - insulin_type_bolus
 - weight
 - gender
 - age_of_diagnosis

Columns being dropped:
 - is_pregnant
 - ethnicity
 - insulin_delivery_device
 - cgm_device
 - is_test
 - source_file
 - subject_split_across_traintest
 - patient_stream_id
Processed row group 2/949
Processed row group 3/949
Processed row group 4/949
Processed row group 5/949
Processed row group 6/949
Processed row group 7/949
Processed row group 8/949
Processed row group 9/949
Processed row group 10/949
Processed row group 11/949
Processed row group 12/949
Processed row group 13/949
Processed row group 14/949
Processed row group 15/949
Processed row group 16/949
Processed row group 17/949
Processed row group 18/949
Processed row group 19/949
Processed row group 20/949
Processed row group 21/949
Processed row group 22/949
Processed row group 23/949
P

In [7]:
import pyarrow.parquet as pq

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/Loop.parquet'

pf = pq.ParquetFile(path)

nan_count = 0
total_count = 0

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["insulin_type_bolus"]
):
    col = batch.column("insulin_type_bolus")
    nan_count += col.null_count
    total_count += len(col)

print("Total rows:", total_count)
print("NaN/Null values:", nan_count)
print("Non-null values:", total_count - nan_count)
print("Missing %:", (nan_count / total_count) * 100)

Total rows: 94810573
NaN/Null values: 9178209
Non-null values: 85632364
Missing %: 9.68057539321063


In [9]:
import pyarrow.parquet as pq
from collections import Counter

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/Loop.parquet'

pf = pq.ParquetFile(path)

values = Counter()

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["insulin_type_bolus"]
):
    for value in batch.column("insulin_type_bolus").to_pylist():
        if value is not None:
            values[value] += 1

print("Number of different values:", len(values))

print("\nDifferent insulin_type_bolus values:")
for value, count in values.most_common():
    print(f"{value}: {count}")

Number of different values: 5

Different insulin_type_bolus values:
Humalog (Lispro): 41615286
Novolog (Aspart): 31837314
Fiasp (Rapid Aspart): 9972427
Apidra (Glusine): 2138319
Regular insulin: 69018


In [2]:
import pyarrow.parquet as pq
import pandas as pd

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/Loopdropped.parquet'


pf = pq.ParquetFile(path)

nan_counts = {col: 0 for col in pf.schema.names}
total_rows = pf.metadata.num_rows

for batch in pf.iter_batches(batch_size=100_000):
    df = batch.to_pandas()

    for col in df.columns:
        nan_counts[col] += df[col].isna().sum()

result = pd.DataFrame({
    "Column": nan_counts.keys(),
    "NaN Count": nan_counts.values(),
})

result["NaN Percentage"] = (
    result["NaN Count"] / total_rows * 100
)

result = result.sort_values("NaN Percentage", ascending=False)

print(f"Total rows: {total_rows:,}")
print("=" * 80)
print(result.to_string(index=False))

Total rows: 94,810,573
                    Column  NaN Count  NaN Percentage
                     carbs   15347962       16.188028
                       CGM   10175566       10.732522
                     bolus    9925768       10.469052
                     basal    9039245        9.534005
                   insulin    8936497        9.425633
                    weight    7732368        8.155597
          age_of_diagnosis    7603314        8.019479
                    height    7487761        7.897601
                    gender    6996601        7.379558
                       age          0        0.000000
                      date          0        0.000000
                        id          0        0.000000
insulin_delivery_algorithm          0        0.000000
 insulin_delivery_modality          0        0.000000


In [4]:
import pyarrow.parquet as pq
import pandas as pd

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/Loopdropped.parquet'

pf = pq.ParquetFile(path)
total_rows = pf.metadata.num_rows

counts = {
    "CGM available": 0,
    "Carbs available": 0,
    "Bolus available": 0,
    "CGM + Carbs available": 0,
    "CGM + Bolus available": 0,
    "Carbs + Bolus available": 0,
    "CGM + Carbs + Bolus available": 0,
}

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["CGM", "carbs", "bolus"]
):
    df = batch.to_pandas()

    cgm = df["CGM"].notna()
    carbs = df["carbs"].notna()
    bolus = df["bolus"].notna()

    counts["CGM available"] += cgm.sum()
    counts["Carbs available"] += carbs.sum()
    counts["Bolus available"] += bolus.sum()

    counts["CGM + Carbs available"] += (cgm & carbs).sum()
    counts["CGM + Bolus available"] += (cgm & bolus).sum()
    counts["Carbs + Bolus available"] += (carbs & bolus).sum()
    counts["CGM + Carbs + Bolus available"] += (cgm & carbs & bolus).sum()

print(f"Total rows: {total_rows:,}")
print("=" * 80)

for name, count in counts.items():
    percentage = count / total_rows * 100
    print(f"{name:<35} {count:>15,}  ({percentage:6.2f}%)")

Total rows: 94,810,573
CGM available                            84,635,007  ( 89.27%)
Carbs available                          79,462,611  ( 83.81%)
Bolus available                          84,884,805  ( 89.53%)
CGM + Carbs available                    71,986,295  ( 75.93%)
CGM + Bolus available                    75,829,099  ( 79.98%)
Carbs + Bolus available                  78,584,700  ( 82.89%)
CGM + Carbs + Bolus available            71,196,002  ( 75.09%)


In [5]:
import pyarrow.parquet as pq
import pandas as pd

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/Loopdropped.parquet'

pf = pq.ParquetFile(path)
total_rows = pf.metadata.num_rows

counts = {
    "carbs_nan": 0,
    "carbs_zero": 0,
    "carbs_positive": 0,
    "bolus_nan": 0,
    "bolus_zero": 0,
    "bolus_positive": 0,
    
    "bolus_zero_insulin_equals_basal": 0,
    "bolus_zero_with_values": 0,
    
    "bolus_positive_insulin_matches_sum": 0,
    "bolus_positive_with_values": 0,
}

for batch in pf.iter_batches(
    batch_size=100_000,
    columns=["carbs", "bolus", "basal", "insulin"]
):
    df = batch.to_pandas()

    # -------------------------
    # CARBS
    # -------------------------
    carbs_nan = df["carbs"].isna()
    carbs_zero = df["carbs"].notna() & (df["carbs"] == 0)
    carbs_positive = df["carbs"] > 0

    counts["carbs_nan"] += carbs_nan.sum()
    counts["carbs_zero"] += carbs_zero.sum()
    counts["carbs_positive"] += carbs_positive.sum()

    # -------------------------
    # BOLUS
    # -------------------------
    bolus_nan = df["bolus"].isna()
    bolus_zero = df["bolus"].notna() & (df["bolus"] == 0)
    bolus_positive = df["bolus"] > 0

    counts["bolus_nan"] += bolus_nan.sum()
    counts["bolus_zero"] += bolus_zero.sum()
    counts["bolus_positive"] += bolus_positive.sum()

    # -------------------------
    # BOLUS = 0
    # Check insulin ≈ basal
    # -------------------------
    valid_zero = (
        bolus_zero &
        df["insulin"].notna() &
        df["basal"].notna()
    )

    counts["bolus_zero_with_values"] += valid_zero.sum()

    counts["bolus_zero_insulin_equals_basal"] += (
        valid_zero &
        ((df["insulin"] - df["basal"]).abs() < 1e-6)
    ).sum()

    # -------------------------
    # BOLUS > 0
    # Check insulin ≈ basal + bolus
    # -------------------------
    valid_positive = (
        bolus_positive &
        df["insulin"].notna() &
        df["basal"].notna()
    )

    counts["bolus_positive_with_values"] += valid_positive.sum()

    counts["bolus_positive_insulin_matches_sum"] += (
        valid_positive &
        ((df["insulin"] - (df["basal"] + df["bolus"])).abs() < 1e-6)
    ).sum()


# ============================================================
# PRINT RESULTS
# ============================================================

print(f"Total rows: {total_rows:,}")
print("=" * 90)

print("\nCARBS")
print("-" * 90)

for key in ["carbs_nan", "carbs_zero", "carbs_positive"]:
    count = counts[key]
    print(f"{key:<25} {count:>15,}  ({count / total_rows * 100:6.2f}%)")


print("\nBOLUS")
print("-" * 90)

for key in ["bolus_nan", "bolus_zero", "bolus_positive"]:
    count = counts[key]
    print(f"{key:<25} {count:>15,}  ({count / total_rows * 100:6.2f}%)")


print("\nBOLUS = 0 → INSULIN vs BASAL")
print("-" * 90)

print(
    f"Rows with bolus=0 and insulin/basal available: "
    f"{counts['bolus_zero_with_values']:,}"
)

print(
    f"Rows where insulin ≈ basal: "
    f"{counts['bolus_zero_insulin_equals_basal']:,}"
)

if counts["bolus_zero_with_values"] > 0:
    pct = (
        counts["bolus_zero_insulin_equals_basal"]
        / counts["bolus_zero_with_values"]
        * 100
    )
    print(f"Percentage: {pct:.2f}%")


print("\nBOLUS > 0 → INSULIN vs BASAL + BOLUS")
print("-" * 90)

print(
    f"Rows with bolus>0 and insulin/basal available: "
    f"{counts['bolus_positive_with_values']:,}"
)

print(
    f"Rows where insulin ≈ basal + bolus: "
    f"{counts['bolus_positive_insulin_matches_sum']:,}"
)

if counts["bolus_positive_with_values"] > 0:
    pct = (
        counts["bolus_positive_insulin_matches_sum"]
        / counts["bolus_positive_with_values"]
        * 100
    )
    print(f"Percentage: {pct:.2f}%")

Total rows: 94,810,573

CARBS
------------------------------------------------------------------------------------------
carbs_nan                      15,347,962  ( 16.19%)
carbs_zero                     78,128,899  ( 82.41%)
carbs_positive                  1,333,712  (  1.41%)

BOLUS
------------------------------------------------------------------------------------------
bolus_nan                       9,925,768  ( 10.47%)
bolus_zero                     82,400,633  ( 86.91%)
bolus_positive                  2,484,172  (  2.62%)

BOLUS = 0 → INSULIN vs BASAL
------------------------------------------------------------------------------------------
Rows with bolus=0 and insulin/basal available: 81,815,042
Rows where insulin ≈ basal: 81,815,042
Percentage: 100.00%

BOLUS > 0 → INSULIN vs BASAL + BOLUS
------------------------------------------------------------------------------------------
Rows with bolus>0 and insulin/basal available: 2,473,609
Rows where insulin ≈ basal + bolus: 2,4

In [2]:
import pyarrow.parquet as pq

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/Loopdropped.parquet'

pf = pq.ParquetFile(path)

print("Rows:", f"{pf.metadata.num_rows:,}")
print("Row groups:", pf.metadata.num_row_groups)

for i in range(min(10, pf.metadata.num_row_groups)):
    rg = pf.metadata.row_group(i)

    print(
        f"Row group {i}: "
        f"{rg.num_rows:,} rows"
    )

Rows: 94,810,573
Row groups: 949
Row group 0: 96,456 rows
Row group 1: 100,000 rows
Row group 2: 100,000 rows
Row group 3: 100,000 rows
Row group 4: 100,000 rows
Row group 5: 100,000 rows
Row group 6: 100,000 rows
Row group 7: 100,000 rows
Row group 8: 100,000 rows
Row group 9: 100,000 rows


In [3]:
import pyarrow.parquet as pq
import pandas as pd
from collections import defaultdict

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/Loopdropped.parquet'

pf = pq.ParquetFile(path)

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------

DROP_THRESHOLDS = [5, 10, 15, 20]   # mg/dL
WINDOWS = [5, 10, 15, 30, 60]       # minutes

# ---------------------------------------------------------
# Counters
# ---------------------------------------------------------

drop_counts = defaultdict(int)
matched_counts = defaultdict(int)

# ---------------------------------------------------------
# Keep only recent events for each patient
#
# We only need recent CGM drops and carb events.
# ---------------------------------------------------------

recent_drops = defaultdict(list)
recent_carbs = defaultdict(list)

# Last CGM reading for each patient
last_cgm = {}

# Last timestamp for each patient
last_time = {}

# ---------------------------------------------------------
# Process row group by row group
# ---------------------------------------------------------

for rg_idx in range(pf.metadata.num_row_groups):

    table = pf.read_row_group(
        rg_idx,
        columns=["id", "date", "CGM", "carbs"]
    )

    df = table.to_pandas()

    df["date"] = pd.to_datetime(df["date"])

    # Sort within row group
    df = df.sort_values(
        ["id", "date"]
    )

    for row in df.itertuples(index=False):

        patient = row.id
        timestamp = row.date
        cgm = row.CGM
        carbs = row.carbs

        # -------------------------------------------------
        # Remove events older than 60 minutes
        # -------------------------------------------------

        cutoff = timestamp - pd.Timedelta(minutes=60)

        recent_drops[patient] = [
            d for d in recent_drops[patient]
            if d[0] >= cutoff
        ]

        recent_carbs[patient] = [
            c for c in recent_carbs[patient]
            if c >= cutoff
        ]

        # -------------------------------------------------
        # Check CGM change
        # -------------------------------------------------

        if (
            patient in last_cgm
            and pd.notna(cgm)
            and pd.notna(last_cgm[patient])
            and patient in last_time
        ):

            previous_cgm = last_cgm[patient]

            change = cgm - previous_cgm

            # Make sure readings are reasonably close
            # in time before calling it a CGM trend
            time_diff = (
                timestamp - last_time[patient]
            ).total_seconds() / 60

            if time_diff > 0 and time_diff <= 10:

                # -------------------------------------------------
                # Record CGM drops
                # -------------------------------------------------

                for threshold in DROP_THRESHOLDS:

                    if change <= -threshold:

                        key = threshold

                        drop_counts[key] += 1

                        # Store drop event
                        recent_drops[patient].append(
                            (timestamp, threshold)
                        )

        # -------------------------------------------------
        # Carb event
        # -------------------------------------------------

        if pd.notna(carbs) and carbs > 0:

            recent_carbs[patient].append(timestamp)

            # -------------------------------------------------
            # Check whether this carb event follows a CGM drop
            # -------------------------------------------------

            for drop_time, threshold in recent_drops[patient]:

                minutes_after = (
                    timestamp - drop_time
                ).total_seconds() / 60

                if minutes_after <= 0:
                    continue

                for window in WINDOWS:

                    if minutes_after <= window:

                        matched_counts[
                            (threshold, window)
                        ] += 1

        # -------------------------------------------------
        # Update previous CGM
        # -------------------------------------------------

        if pd.notna(cgm):

            last_cgm[patient] = cgm
            last_time[patient] = timestamp

    # Progress
    if (rg_idx + 1) % 50 == 0:
        print(
            f"Processed row groups: "
            f"{rg_idx + 1}/{pf.metadata.num_row_groups}"
        )

# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------

print("\n")
print("=" * 100)
print("CGM DROP → CARB EVENT ANALYSIS")
print("=" * 100)

print(
    f"{'Drop threshold':<20}"
    f"{'Window':<15}"
    f"{'Drop events':<20}"
    f"{'Followed by carbs':<20}"
    f"{'%':<10}"
)

print("-" * 100)

for threshold in DROP_THRESHOLDS:

    for window in WINDOWS:

        drops = drop_counts[threshold]

        # matched_counts counts drop-carb pairs,
        # so we need to be careful about duplicate carb events.
        matched = matched_counts[(threshold, window)]

        percentage = (
            matched / drops * 100
            if drops > 0 else 0
        )

        print(
            f"{threshold:>5} mg/dL{'':<8}"
            f"{window:>3} min{'':<9}"
            f"{drops:>15,}"
            f"{matched:>19,}"
            f"{percentage:>9.2f}%"
        )

Processed row groups: 50/949
Processed row groups: 100/949
Processed row groups: 150/949
Processed row groups: 200/949
Processed row groups: 250/949
Processed row groups: 300/949
Processed row groups: 350/949
Processed row groups: 400/949
Processed row groups: 450/949
Processed row groups: 500/949
Processed row groups: 550/949
Processed row groups: 600/949
Processed row groups: 650/949
Processed row groups: 700/949
Processed row groups: 750/949
Processed row groups: 800/949
Processed row groups: 850/949
Processed row groups: 900/949


CGM DROP → CARB EVENT ANALYSIS
Drop threshold      Window         Drop events         Followed by carbs   %         
----------------------------------------------------------------------------------------------------
    5 mg/dL          5 min              12,325,035            217,251     1.76%
    5 mg/dL         10 min              12,325,035            438,364     3.56%
    5 mg/dL         15 min              12,325,035            662,009     5.37%
 

In [4]:
import pyarrow.parquet as pq
import pandas as pd
from collections import defaultdict

path = '/Users/ahmed/Desktop/grad sakr/datasets/metabonet_split/Loopdropped.parquet'

pf = pq.ParquetFile(path)

# =========================================================
# SETTINGS
# =========================================================

BATCH_SIZE = 100_000

DROP_THRESHOLDS = [5, 10, 15, 20]   # mg/dL
WINDOWS = [5, 10, 15, 30, 60]       # minutes

# =========================================================
# Counters
# =========================================================

drop_counts = defaultdict(int)

# Each set stores UNIQUE drop events that were followed
# by at least one carb event.
#
# Key = (patient, drop timestamp)
matched = {
    (threshold, window): set()
    for threshold in DROP_THRESHOLDS
    for window in WINDOWS
}

# Last CGM reading for each patient
last_cgm = {}

# Last CGM timestamp for each patient
last_cgm_time = {}

# Store recent drop events for each patient
# Format:
# [(drop_time, threshold), ...]
recent_drops = defaultdict(list)

# =========================================================
# PROCESS ROW GROUPS
# =========================================================

for rg_idx in range(pf.metadata.num_row_groups):

    table = pf.read_row_group(
        rg_idx,
        columns=["id", "date", "CGM", "carbs"]
    )

    df = table.to_pandas()

    df["date"] = pd.to_datetime(df["date"])

    # Sort this row group chronologically per patient
    df = df.sort_values(["id", "date"])

    for row in df.itertuples(index=False):

        patient = row.id
        timestamp = row.date
        cgm = row.CGM
        carbs = row.carbs

        # =================================================
        # 1. REMOVE OLD DROP EVENTS
        # =================================================

        cutoff = timestamp - pd.Timedelta(minutes=max(WINDOWS))

        recent_drops[patient] = [
            d for d in recent_drops[patient]
            if d[0] >= cutoff
        ]

        # =================================================
        # 2. CHECK WHETHER THIS IS A CARB EVENT
        # =================================================

        if pd.notna(carbs) and carbs > 0:

            for drop_time, threshold in recent_drops[patient]:

                minutes_after = (
                    timestamp - drop_time
                ).total_seconds() / 60

                if minutes_after <= 0:
                    continue

                for window in WINDOWS:

                    if minutes_after <= window:

                        # UNIQUE drop event
                        matched[(threshold, window)].add(
                            (patient, drop_time)
                        )

        # =================================================
        # 3. CHECK FOR CGM DROP
        # =================================================

        if (
            pd.notna(cgm)
            and patient in last_cgm
            and pd.notna(last_cgm[patient])
            and patient in last_cgm_time
        ):

            previous_cgm = last_cgm[patient]

            change = cgm - previous_cgm

            time_diff = (
                timestamp - last_cgm_time[patient]
            ).total_seconds() / 60

            # Only compare reasonably close CGM readings
            if 0 < time_diff <= 10:

                for threshold in DROP_THRESHOLDS:

                    if change <= -threshold:

                        drop_counts[threshold] += 1

                        recent_drops[patient].append(
                            (timestamp, threshold)
                        )

        # =================================================
        # 4. UPDATE LAST CGM
        # =================================================

        if pd.notna(cgm):

            last_cgm[patient] = cgm
            last_cgm_time[patient] = timestamp

    # Progress
    if (rg_idx + 1) % 50 == 0:
        print(
            f"Processed row groups: "
            f"{rg_idx + 1}/{pf.metadata.num_row_groups}"
        )

# =========================================================
# RESULTS
# =========================================================

print("\n")
print("=" * 100)
print("UNIQUE CGM DROP → CARB EVENT ANALYSIS")
print("=" * 100)

print(
    f"{'CGM Drop':<18}"
    f"{'Window':<12}"
    f"{'Drop Events':>18}"
    f"{'Drops → Carbs':>20}"
    f"{'Percentage':>15}"
)

print("-" * 100)

for threshold in DROP_THRESHOLDS:

    for window in WINDOWS:

        total_drops = drop_counts[threshold]

        matched_drops = len(
            matched[(threshold, window)]
        )

        percentage = (
            matched_drops / total_drops * 100
            if total_drops > 0 else 0
        )

        print(
            f"{threshold:>3} mg/dL{'':<10}"
            f"{window:>3} min{'':<5}"
            f"{total_drops:>18,}"
            f"{matched_drops:>20,}"
            f"{percentage:>14.2f}%"
        )

Processed row groups: 50/949
Processed row groups: 100/949
Processed row groups: 150/949
Processed row groups: 200/949
Processed row groups: 250/949
Processed row groups: 300/949
Processed row groups: 350/949
Processed row groups: 400/949
Processed row groups: 450/949
Processed row groups: 500/949
Processed row groups: 550/949
Processed row groups: 600/949
Processed row groups: 650/949
Processed row groups: 700/949
Processed row groups: 750/949
Processed row groups: 800/949
Processed row groups: 850/949
Processed row groups: 900/949


UNIQUE CGM DROP → CARB EVENT ANALYSIS
CGM Drop          Window             Drop Events       Drops → Carbs     Percentage
----------------------------------------------------------------------------------------------------
  5 mg/dL            5 min             12,325,035             217,251          1.76%
  5 mg/dL           10 min             12,325,035             429,618          3.49%
  5 mg/dL           15 min             12,325,035             636,